In [4]:
"""
Global Analysis Data Integration Pipeline
==========================================
This script integrates multiple data sources (World Bank, Disaster, MPI, Vulnerability)
using fuzzy country name matching to create a unified dataset for analysis.

Author: World Bank Group
Date: 2025-01-26
"""

import pandas as pd
import os
import numpy as np
from rapidfuzz import fuzz, process
from rapidfuzz.utils import default_process
import re
import json

# ============================================================================
# FILE PATHS CONFIGURATION
# ============================================================================

folder_path = r"C:\Users\wb512463\OneDrive - WBG\Resilient Housing\Global analysis\data_processed"

# Base World Bank reference data file
file_wb_base = os.path.join(folder_path, "Country_WorldBank_Data_Cleaned_For_Merge.csv")

# Input data files
file_disaster = os.path.join(folder_path, "country_disaster_summary_FINAL_PERIOD_SPLIT.csv")
file_mpi = os.path.join(folder_path, "country_MPI.csv")
file_vulnerability = os.path.join(folder_path, "country_vulnerability_class_shares.csv")

# ============================================================================
# FUZZY MATCHING CONFIGURATION
# ============================================================================

SCORE_THRESHOLD = 90

# Prohibited automatic matching pairs (prevent false matches)
PROHIBITED_PAIRS = [
    ('Arunachal Pradesh', 'Aruba'), ('Aruba', 'Arunachal Pradesh'), 
    ('Burundi', 'Brunei Darussalam'), ('Brunei', 'Burundi'), 
    ('Guatemala', 'Malta'), ('Malta', 'Guatemala'),
    ('Guinea', 'Equatorial Guinea'), ('Equatorial Guinea', 'Guinea'), 
    ('Qatar', 'Gibraltar'), ('Gibraltar', 'Qatar'),
    ('Taiwan', 'Tanzania'), ('Taiwan', 'Togo'),
]

# Country name standardization mapping
STANDARDIZATION_MAP = {
    "Viet Nam": "Viet Nam", "Vietnam": "Viet Nam", 
    "Côte d'Ivoire": "Cote d'Ivoire", "Ivory Coast": "Cote d'Ivoire",
    "Lao PDR": "Lao PDR", "Laos": "Lao PDR", "Lao People's Democratic Republic": "Lao PDR",
    "The former Yugoslav Republic of Macedonia": "North Macedonia", "North_Macedonia": "North Macedonia", 
    "Turkey": "Turkiye", 
    "Russia": "Russian Federation",
    "South Korea": "Korea, Rep.", "Republic of Korea": "Korea, Rep.", 
    "North Korea": "Korea, Dem. People's Rep.", "Dem People's Rep of Korea": "Korea, Dem. People's Rep.",
    "United Republic of Tanzania": "Tanzania",
    "Swaziland": "Eswatini",
    "Slovakia": "Slovak Republic",
    "Syria": "Syrian Arab Republic",
    "Congo": "Congo, Rep.",
    "Democratic Republic of the Congo": "Congo, Dem. Rep.", "Congo, Democratic Republic of the": "Congo, Dem. Rep.",
    "U.K. of Great Britain and Northern Ireland": "United Kingdom",
    "United States of America": "United States",
    "Venezuela": "Venezuela, RB",
    "US Virgin Islands": "Virgin Islands (U.S.)",
    "Saint Kitts and Nevis": "St. Kitts and Nevis",
    "Saint Lucia": "St. Lucia",
    "Saint Vincent and the Grenadines": "St. Vincent and the Grenadines",
    "Palestine, State of": "West Bank and Gaza", "Palestine": "West Bank and Gaza",
    "Bahamas": "Bahamas, The",
    "Cape Verde": "Cabo Verde",
    "Czech Republic": "Czechia",
    "Egypt": "Egypt, Arab Rep.",
    "Gambia": "Gambia, The",
    "Hong Kong": "Hong Kong SAR, China",
    "Iran  (Islamic Republic of)": "Iran, Islamic Rep.",
    "Iran": "Iran, Islamic Rep.",
    "IRAN (ISLAMIC REPUBLIC OF)": "Iran, Islamic Rep.",
    "Kyrgyzstan": "Kyrgyz Republic",
    "Macau": "Macao SAR, China",
    "Macao": "Macao SAR, China",
    "Micronesia (Federated States of)": "Micronesia, Fed. Sts.",
    "Micronesia": "Micronesia, Fed. Sts.",
    "Moldova, Republic of": "Moldova",
    "Puerto Rico": "Puerto Rico (US)",
    "United States Virgin Islands": "Virgin Islands (U.S.)",
    "Brunei": "Brunei Darussalam", 
    "Yemen": "Yemen, Rep.",
    "Somalia": "Somalia, Fed. Rep.",
    "Chad": "Chad", 
    "Gibraltar": "Gibraltar",
    "Cook Islands": "Cook Islands", 
    "Montserrat": "Montserrat",
    "Anguilla": "Anguilla",
    "Niue": "Niue",    
    "Taiwan": "Taiwan",
    "Bolivia (Plurinational State of)": "Bolivia", 
    "China, Hong Kong Special Administrative Region": "Hong Kong SAR, China",
    "Iran (Islamic Republic of)": "Iran, Islamic Rep.",
    "Democratic People's Republic of Korea": "Korea, Dem. People's Rep.",
    "China, Macao Special Administrative Region": "Macao SAR, China",
    "Saint Martin (French Part)": "St. Martin (French part)",
    "Türkiye": "Turkiye",
    "United Kingdom of Great Britain and Northern Ireland": "United Kingdom",
    "Venezuela (Bolivarian Republic of)": "Venezuela, RB",
    "State of Palestine": "West Bank and Gaza",
    "Netherlands (Kingdom of the)": "Netherlands",
    "Serbia Montenegro": "Serbia Montenegro", 
    "Taiwan (Province of China)": "Taiwan", 
    "Yemen Arab Republic": "Yemen Arab Republic", 
    "People's Democratic Republic of Yemen": "People's Democratic Republic of Yemen",
    "Germany Federal Republic": "Germany", 
    "Canary Islands": "Canary Islands",
    "French Guiana": "French Guiana",
    "Guadeloupe": "Guadeloupe",
    "Martinique": "Martinique",
    "Réunion": "Réunion",
    "Wallis and Futuna Islands": "Wallis and Futuna Islands",
    "CÔTE D'IVOIRE": "Cote d'Ivoire",
    "REPUBLIC OF MOLDOVA": "Moldova",
}

# ============================================================================
# DATA CLEANING FUNCTIONS
# ============================================================================

def clean_country_names(df):
    """
    Clean and standardize country names in the dataframe.
    
    Args:
        df (pd.DataFrame): Input dataframe with country names as index.
    
    Returns:
        pd.DataFrame: Cleaned dataframe with standardized country names.
    """
    df_reset = df.reset_index()
    index_name = df_reset.columns[0]
    
    # Convert to string and normalize whitespace
    df_reset[index_name] = df_reset[index_name].astype(str)    
    df_reset[index_name] = df_reset[index_name].str.replace(r'\s+', ' ', regex=True)
    df_reset[index_name] = df_reset[index_name].str.strip()
    
    # Apply standardization mapping (case-insensitive)
    standardization_map_lower = {k.lower(): v.lower() for k, v in STANDARDIZATION_MAP.items()}
    temp_series_lower = df_reset[index_name].str.lower().replace(standardization_map_lower)    
    df_reset[index_name] = temp_series_lower.str.upper()

    # Aggregate duplicate countries after standardization
    grouped = df_reset.groupby(index_name)
    df_numeric = grouped.sum(numeric_only=True)
    df_non_numeric = grouped.first(numeric_only=False).select_dtypes(include=['object', 'bool'])
    df_cleaned = df_numeric.merge(df_non_numeric, left_index=True, right_index=True, how='outer')
    
    return df_cleaned


def aggregate_disaster_data(df):
    """
    Aggregate Gaza Strip and West Bank into 'West Bank and Gaza'.
    
    Args:
        df (pd.DataFrame): Disaster data with country names as index.
    
    Returns:
        pd.DataFrame: Aggregated disaster data.
    """
    countries_to_sum = ['Gaza Strip', 'West Bank']
    
    if not all(c in df.index for c in countries_to_sum): 
        return df
    
    new_palestine_data = df.loc[countries_to_sum].sum(axis=0, numeric_only=True)
    df.loc['West Bank and Gaza'] = new_palestine_data
    df = df.drop(index=countries_to_sum)
    
    return df


# ============================================================================
# FUZZY MATCHING FUNCTION
# ============================================================================

def fuzzy_match_country_v2(country_name, choices, threshold=SCORE_THRESHOLD):
    """
    Perform fuzzy matching of country names with prohibited pairs filtering.
    
    Args:
        country_name (str): Country name to match.
        choices (list): List of reference country names.
        threshold (int): Minimum matching score (default: 90).
    
    Returns:
        tuple: (matched_country_name, match_score) or (None, 0) if no match.
    """
    if pd.isna(country_name): 
        return (None, 0)
    if country_name in choices: 
        return (country_name, 100)
    
    match = process.extractOne(
        country_name, 
        choices, 
        scorer=fuzz.ratio, 
        score_cutoff=threshold, 
        processor=default_process 
    )
    
    if match:
        matched_country = match[0]
        score = match[1]
        
        # Check prohibited pairs
        if (country_name, matched_country) in PROHIBITED_PAIRS:
            return (None, 0)
            
        if score >= 95:
            return (matched_country, score)
        return (None, score) 
    else:
        return (None, 0)


# ============================================================================
# DATA LOADING AND INITIAL PROCESSING
# ============================================================================

print("\n" + "="*70)
print("GLOBAL ANALYSIS DATA INTEGRATION PIPELINE")
print("="*70 + "\n")

# Load World Bank base data (reference standard)
df_wb_base = pd.read_csv(file_wb_base).rename(columns={'CountryName': 'CountryName'}).set_index('CountryName')
base_countries = df_wb_base.index.tolist()
print(f"Reference country list (World Bank): {len(base_countries)} countries")

# 1. Load and process Disaster data
df_disaster = pd.read_csv(file_disaster).rename(columns={'country_name': 'CountryName', 'CountryName': 'CountryName'}).set_index('CountryName')
df_disaster = aggregate_disaster_data(df_disaster)
df_disaster = clean_country_names(df_disaster)

# 2. Load and process MPI data
df_mpi = pd.read_csv(file_mpi).rename(columns={'Country': 'CountryName'}).set_index('CountryName')
df_mpi = clean_country_names(df_mpi)

# 3. Load and process Vulnerability data - CORRECTED VERSION
# Simply read the CSV with default settings - the file already has proper headers
df_vulnerability = pd.read_csv(file_vulnerability)

# Identify the country name column (could be 'country', 'CountryName', etc.)
country_col = None
for col in df_vulnerability.columns:
    if col.lower() in ['country', 'countryname', 'country_name']:
        country_col = col
        break

if country_col is None:
    # If no country column found, assume first column is country
    country_col = df_vulnerability.columns[0]

# Rename and set index
df_vulnerability = df_vulnerability.rename(columns={country_col: 'CountryName'}).set_index('CountryName')

# Drop any unnecessary columns (like iso3 if it exists)
if 'iso3' in df_vulnerability.columns:
    df_vulnerability = df_vulnerability.drop(columns=['iso3'])

# Apply country name cleaning
df_vulnerability = clean_country_names(df_vulnerability)

# Debug output - indicator counts after cleaning
print("\n--- 📊 Indicator Counts After Cleaning ---")
print(f"1. WB (Reference): {len(df_wb_base.columns)} columns")
print(f"2. Disaster: {len(df_disaster.columns)} columns")
print(f"3. MPI: {len(df_mpi.columns)} columns")
print(f"4. Vulnerability: {len(df_vulnerability.columns)} columns")


# ============================================================================
# FUZZY MATCHING AND DATA INTEGRATION
# ============================================================================

print("\n--- 🔗 Performing Fuzzy Matching ---")

# Disaster matching
df_disaster_reset = df_disaster.reset_index()
df_disaster_reset[['Matched_CountryName', 'Match_Score_DIS']] = df_disaster_reset['CountryName'].apply(
    lambda x: pd.Series(fuzzy_match_country_v2(x, base_countries))
)
df_disaster_matched = df_disaster_reset[df_disaster_reset['Matched_CountryName'].notna()].set_index('Matched_CountryName').drop(columns=['CountryName'])

# MPI matching
df_mpi_reset = df_mpi.reset_index()
df_mpi_reset[['Matched_CountryName', 'Match_Score_MPI']] = df_mpi_reset['CountryName'].apply(
    lambda x: pd.Series(fuzzy_match_country_v2(x, base_countries))
)
df_mpi_matched = df_mpi_reset[df_mpi_reset['Matched_CountryName'].notna()].set_index('Matched_CountryName').drop(columns=['CountryName'])

# Vulnerability matching
df_vuln_reset = df_vulnerability.reset_index()
df_vuln_reset[['Matched_CountryName', 'Match_Score_VULN']] = df_vuln_reset['CountryName'].apply(
    lambda x: pd.Series(fuzzy_match_country_v2(x, base_countries))
)
df_vuln_matched = df_vuln_reset[df_vuln_reset['Matched_CountryName'].notna()].set_index('Matched_CountryName').drop(columns=['CountryName'])

# Final data integration (left join on World Bank reference)
df_final_merged_fuzzy = df_wb_base.copy()
df_final_merged_fuzzy = df_final_merged_fuzzy.merge(df_disaster_matched, left_index=True, right_index=True, how='left')
df_final_merged_fuzzy = df_final_merged_fuzzy.merge(df_mpi_matched, left_index=True, right_index=True, how='left')
df_final_merged_fuzzy = df_final_merged_fuzzy.merge(df_vuln_matched, left_index=True, right_index=True, how='left')


# ============================================================================
# CREATE TRACEABILITY INFORMATION
# ============================================================================

print("\n--- 📝 Creating Traceability Information ---")

# Create trace dataframe
df_trace = df_final_merged_fuzzy.index.to_frame(name='Merged_CountryName')

df_disaster_trace = df_disaster_reset[['CountryName', 'Matched_CountryName', 'Match_Score_DIS']].rename(columns={'CountryName': 'DISASTER_Original_Name'})
df_mpi_trace = df_mpi_reset[['CountryName', 'Matched_CountryName', 'Match_Score_MPI']].rename(columns={'CountryName': 'MPI_Original_Name'})
df_vuln_trace = df_vuln_reset[['CountryName', 'Matched_CountryName', 'Match_Score_VULN']].rename(columns={'CountryName': 'VULN_Original_Name'})

# Merge trace information
df_trace = df_trace.merge(df_disaster_trace, left_on='Merged_CountryName', right_on='Matched_CountryName', how='left').drop(columns=['Matched_CountryName']).drop_duplicates(subset=['Merged_CountryName'], keep='first')
df_trace = df_trace.merge(df_mpi_trace, left_on='Merged_CountryName', right_on='Matched_CountryName', how='left').drop(columns=['Matched_CountryName']).drop_duplicates(subset=['Merged_CountryName'], keep='first')
df_trace = df_trace.merge(df_vuln_trace, left_on='Merged_CountryName', right_on='Matched_CountryName', how='left').drop(columns=['Matched_CountryName']).drop_duplicates(subset=['Merged_CountryName'], keep='first')

df_trace['WB_Original_Name'] = df_trace['Merged_CountryName']
df_trace['WB_Match_Score'] = 100.0

# Create final export dataframe
df_trace = df_trace.set_index('Merged_CountryName')
df_export = df_trace.merge(df_final_merged_fuzzy, left_index=True, right_index=True, how='right')
df_export = df_export.reset_index().rename(columns={'index': 'Merged_CountryName'})


# ============================================================================
# DATA CLEANING AND TYPE CONVERSION
# ============================================================================

print("\n--- 🧹 Final Data Cleaning ---")

# Drop redundant columns
columns_to_drop = [
    'ISO country numeric code', 'ISO country code',
    'ISO_CODE', 'iso3', 
]
cols_to_drop_present = [col for col in columns_to_drop if col in df_export.columns]
df_analysis = df_export.drop(columns=cols_to_drop_present, errors='ignore')


# ============================================================================
# INTEGRATION RESULTS SUMMARY
# ============================================================================

# Calculate statistics
wb_count = len(base_countries)
disaster_original_count = len(df_disaster.index)
disaster_matched_count = len(df_disaster_matched)
mpi_original_count = len(df_mpi.index)
mpi_matched_count = len(df_mpi_matched)
vuln_original_count = len(df_vulnerability.index)
vuln_matched_count = len(df_vuln_matched)

# Unmatched countries
disaster_unmatched_list = df_disaster_reset[df_disaster_reset['Matched_CountryName'].isna()]['CountryName'].tolist()
mpi_unmatched_list = df_mpi_reset[df_mpi_reset['Matched_CountryName'].isna()]['CountryName'].tolist()
vuln_unmatched_list = df_vuln_reset[df_vuln_reset['Matched_CountryName'].isna()]['CountryName'].tolist()

# Print results
print("\n" + "="*70)
print("             🌍 DATA INTEGRATION RESULTS SUMMARY 🌍")
print("="*70)
print(f"**World Bank Reference Country Count:** {wb_count} countries\n")

# Disaster
print("--- 1. Disaster Data ---")
print(f"Original data country count (after standardization): {disaster_original_count}")
print(f"Successfully matched countries: {disaster_matched_count}")
print(f"Matching rate: {disaster_matched_count / disaster_original_count * 100:.2f}%")
print(f"Unmatched countries: {disaster_original_count - disaster_matched_count}")
if disaster_unmatched_list:
    print("\n🔥 Unmatched country list (Disaster):")
    print(disaster_unmatched_list)
print("-" * 70)

# MPI
print("\n--- 2. MPI Data ---")
print(f"Original data country count (after standardization): {mpi_original_count}")
print(f"Successfully matched countries: {mpi_matched_count}")
print(f"Matching rate: {mpi_matched_count / mpi_original_count * 100:.2f}%")
print(f"Unmatched countries: {mpi_original_count - mpi_matched_count}")
if mpi_unmatched_list:
    print("\n🔥 Unmatched country list (MPI):")
    print(mpi_unmatched_list)
print("-" * 70)

# Vulnerability
print("\n--- 3. Vulnerability Data ---")
print(f"Original data country count (after standardization): {vuln_original_count}")
print(f"Successfully matched countries: {vuln_matched_count}")
print(f"Matching rate: {vuln_matched_count / vuln_original_count * 100:.2f}%")
print(f"Unmatched countries: {vuln_original_count - vuln_matched_count}")
if vuln_unmatched_list:
    print("\n🔥 Unmatched country list (Vulnerability):")
    print(vuln_unmatched_list)
print("-" * 70)

print("\n" + "="*70)
print(f"**Final integrated dataset row count:** {len(df_final_merged_fuzzy)} rows (World Bank reference)")
print("="*70)


# ============================================================================
# COLUMN NAME CLEANING
# ============================================================================

def clean_dataframe_columns(df):
    """
    Remove unnecessary suffixes and apply consistent prefixes to column names.
    
    Args:
        df (pd.DataFrame): Input dataframe.
    
    Returns:
        pd.DataFrame: Dataframe with cleaned column names.
    """
    new_columns_map = {}
    
    # Pattern to remove unwanted suffixes
    UNWANTED_SUFFIX_PATTERN = r'_UNNAMED:\_\d+$|_UNNAMED'
    
    for col in df.columns:
        new_col = col.upper()
        
        # 1. Remove unnecessary suffixes
        new_col = re.sub(UNWANTED_SUFFIX_PATTERN, '', new_col)
        
        # 2. Replace special characters
        new_col = new_col.replace(' ', '_').replace('-', '_').replace(':', '').replace('.', '_').replace('__', '_')
        
        # 3. Apply prefixes and clear naming conventions
        if new_col == 'MERGED_COUNTRYNAME' or new_col == 'COUNTRYNAME':
            new_col = 'COUNTRY_NAME'
        elif new_col == 'COUNTRY_CODE':
            new_col = 'WB_COUNTRY_CODE'

        # World Bank economic indicators
        elif new_col.startswith(('GDP_', 'POVERTY_', 'GINI_', 'SLUM_POPULATION_', 'WGI_', 'DB_', 'POPULATION_')):
            if not new_col.startswith('WB_'):
                new_col = 'WB_' + new_col
        elif new_col == 'GNI_PC':
            new_col = 'WB_GNI_PC'
        
        # MPI indicators
        elif new_col in ['MPI', 'HEADCOUT_RATIO', 'INTENSITY_OF_DEPREVIATION', 'POPULATION_2023_K', 'YEARS_OF_SCHOOLING', 
                         'SANITATION', 'DRINKING_WATER', 'ELECTRICITY', 'ASSETS', 'NUTRITION', 
                         'CHILD_MORTALITY', 'SCHOOL_ATTENDANCE', 'COOKING_FUEL', 'HOUSING', 
                         'MONETARY_POVERTY_USD3', 'HID', 'WORLD_REGION', 'SURVEY', 'YEAR', 'INCOME_CATEGORY']:
            if not new_col.startswith('MPI_'):
                new_col = 'MPI_' + new_col
            if new_col == 'MPI_MONETARY_POVERTY_USD3':
                new_col = 'MPI_POV_USD3'
            
        # Vulnerability indicators - keep as they come from the CSV
        elif any(hazard in new_col for hazard in ['EARTHQUAKE', 'WIND', 'FLOOD', 'STORM_SURGE', 'TSUNAMI', 'WATER', 'DEFAULT']):
            if not new_col.startswith('VULN_'):
                new_col = 'VULN_' + new_col
            
        # Disaster indicators
        elif new_col.startswith('DIS_'):
            pass

        # Final duplicate check
        if new_col in new_columns_map.values():
            new_col += '_DUPLICATE'

        new_columns_map[col] = new_col

    df = df.rename(columns=new_columns_map)
    return df


# ============================================================================
# PROCESS AND EXPORT DATA
# ============================================================================

print("\n--- 🔧 Processing Column Names ---")

# Drop matching-related columns
COLUMNS_TO_DROP_FOR_ANALYSIS = [
    "WB_Original_Name", "WB_Match_Score", "ISO_CODE",
    "DISASTER_Original_Name", "Match_Score_DIS", "Match_Score_DIS_x", "Match_Score_DIS_y",
    "MPI_Original_Name", "Match_Score_MPI", "Match_Score_MPI_x", "Match_Score_MPI_y",
    "VULN_Original_Name", "Match_Score_VULN", "Match_Score_VULN_x", "Match_Score_VULN_y",
]

cols_to_drop_present = [col for col in COLUMNS_TO_DROP_FOR_ANALYSIS if col in df_analysis.columns]
df_final_for_analysis = df_analysis.drop(columns=cols_to_drop_present, errors='ignore')

# Apply column name cleaning
df_final_for_analysis = clean_dataframe_columns(df_final_for_analysis)

print(f"✅ Column name cleaning complete. Total columns: {len(df_final_for_analysis.columns)}")

# Display sample of VULN column names to verify they're correct
vuln_cols_sample = [col for col in df_final_for_analysis.columns if col.startswith('VULN_')][:10]
print(f"\n📋 Sample VULN column names (first 10):")
for col in vuln_cols_sample:
    print(f"   • {col}")


# ============================================================================
# EXPORT FINAL CSV
# ============================================================================

print("\n--- 💾 Exporting Final CSV ---")

export_final_analysis_path = os.path.join(folder_path, "global_analysis_final_for_modeling.csv")
df_final_for_analysis.to_csv(export_final_analysis_path, index=False, encoding='utf-8-sig')

print(f"✅ Final CSV for analysis and modeling has been exported.")
print(f"   Path: {export_final_analysis_path}")
print(f"   Final data shape: {df_final_for_analysis.shape}")


# ============================================================================
# CREATE AND EXPORT METADATA JSON
# ============================================================================

print("\n--- 📝 Creating Metadata JSON ---")

metadata_file_path_final = os.path.join(folder_path, "column_metadata_final_for_analysis.json")

COLUMN_METADATA_FINAL = {
    # Country identification
    "COUNTRY_NAME": "Country Name (World Bank Standardized Name) - Key Column",
    "WB_COUNTRY_CODE": "Country Code (e.g., USA, JPN) - Categorical",
    
    # World Bank economic indicators
    "WB_GDP_PPP_INTL_2021_VALUE": "GDP (PPP 2021 US$)",
    "WB_GDP_PPP_INTL_2021_YEAR": "GDP (PPP 2021 US$) - Latest Year",
    "WB_GDP_PCAP_PPP_INTL_2021_VALUE": "GDP per capita (PPP 2021 US$)",
    "WB_GDP_PCAP_PPP_INTL_2021_YEAR": "GDP per capita (PPP 2021 US$) - Latest Year",
    "WB_GNI_PC": "Gross National Income per capita (WB Source)",
    "WB_POVERTY_SURVEY_MEAN_INCOME_PPP_VALUE": "Poverty Survey Mean Income (PPP) - Value",
    "WB_POVERTY_SURVEY_MEAN_INCOME_PPP_YEAR": "Poverty Survey Mean Income (PPP) - Year",
    "WB_POVERTY_HC_RATIO_AT_USD8_30_VALUE": "Poverty Headcount Ratio at $8.30 PPP/day - Value",
    "WB_POVERTY_HC_RATIO_AT_USD8_30_YEAR": "Poverty Headcount Ratio at $8.30 PPP/day - Year",
    "WB_POVERTY_HC_RATIO_NATIONAL_LINE_VALUE": "Poverty Headcount Ratio at National Line - Value",
    "WB_POVERTY_HC_RATIO_NATIONAL_LINE_YEAR": "Poverty Headcount Ratio at National Line - Year",
    "WB_POVERTY_HC_RATIO_AT_USD4_20_VALUE": "Poverty Headcount Ratio at $4.20 PPP/day - Value",
    "WB_POVERTY_HC_RATIO_AT_USD4_20_YEAR": "Poverty Headcount Ratio at $4.20 PPP/day - Year",
    "WB_GINI_INDEX_VALUE": "Gini Index (Inequality Measure) - Value",
    "WB_GINI_INDEX_YEAR": "Gini Index - Year",
    "WB_POVERTY_GAP_AT_USD3_00_VALUE": "Poverty Gap at $3.00 PPP/day - Value",
    "WB_POVERTY_GAP_AT_USD3_00_YEAR": "Poverty Gap at $3.00 PPP/day - Year",
    "WB_SLUM_POPULATION_URBAN_PCT_VALUE": "Slum Population as Percentage of Urban Population - Value",
    "WB_SLUM_POPULATION_URBAN_PCT_YEAR": "Slum Population as Percentage of Urban Population - Year",
    "WB_WGI_VOICE_ACCOUNTABILITY_VALUE": "WGI Voice and Accountability - Value",
    "WB_WGI_POLITICAL_STABILITY_VALUE": "WGI Political Stability - Value",
    "WB_WGI_GOVERNMENT_EFFECTIVENESS_VALUE": "WGI Government Effectiveness - Value",
    "WB_WGI_REGULATORY_QUALITY_VALUE": "WGI Regulatory Quality - Value",
    "WB_WGI_RULE_OF_LAW_VALUE": "WGI Rule of Law - Value",
    "WB_WGI_CONTROL_OF_CORRUPTION_VALUE": "WGI Control of Corruption - Value",
    "WB_WGI_VOICE_ACCOUNTABILITY_YEAR": "WGI Voice and Accountability - Year",
    "WB_WGI_POLITICAL_STABILITY_YEAR": "WGI Political Stability - Year",
    "WB_WGI_GOVERNMENT_EFFECTIVENESS_YEAR": "WGI Government Effectiveness - Year",
    "WB_WGI_REGULATORY_QUALITY_YEAR": "WGI Regulatory Quality - Year",
    "WB_WGI_RULE_OF_LAW_YEAR": "WGI Rule of Law - Year",
    "WB_WGI_CONTROL_OF_CORRUPTION_YEAR": "WGI Control of Corruption - Year",
    "WB_DB_CONSTRUCTION_PERMITS_SCORE": "Doing Business Construction Permit Score",
    "WB_DB_BUILDING_QUALITY_INDEX": "Doing Business Building Quality Index",
    "WB_POPULATION_TOTAL_YEAR": "Country Population - Year",
    "WB_POPULATION_TOTAL_VALUE": "Country Population",

    # MPI indicators
    "MPI_MPI": "Multi-dimensional Poverty Index - 2025",
    "MPI_HEADCOUT_RATIO": "Proportion of population who are multidimensionally poor - %",
    "MPI_INTENSITY_OF_DEPREVIATION": "Intensity of deprivation among the poor - %",
    "MPI_POPULATION_2023_K": "2023 Population (Unit: thousands) - Integer",
    "MPI_YEARS_OF_SCHOOLING": "Percentage of people deprived in years of schooling - %",
    "MPI_SANITATION": "Percentage of people deprived in sanitation - %",
    "MPI_DRINKING_WATER": "Percentage of people deprived in drinking water - %",
    "MPI_ELECTRICITY": "Percentage of people deprived in electricity - %",
    "MPI_ASSETS": "Percentage of people deprived in assets - %",
    "MPI_NUTRITION": "Percentage of people deprived in nutrition - %",
    "MPI_CHILD_MORTALITY": "Percentage of people deprived in child mortality - %",
    "MPI_SCHOOL_ATTENDANCE": "Percentage of people deprived in school attendance - %",
    "MPI_COOKING_FUEL": "Percentage of people deprived in cooking fuel - %",
    "MPI_HOUSING": "Percentage of people deprived in housing - %",
    "MPI_POV_USD3": "Poverty rate at $3 a day (MPI Source) - %",
    "MPI_HID": "Human Development Indicator - Value between 0.00 and 1.00",
    "MPI_WORLD_REGION": "World Region (e.g., East Asia) - Categorical Data",
    "MPI_SURVEY": "Name of the survey used for the MPI source - Categorical Data",
    "MPI_YEAR": "Calendar year of the survey used for the MPI source - Categorical Data",
    "MPI_INCOME_CATEGORY": "Country Income Category (e.g., Low Income) - Categorical Data",

    # Disaster indicators (2000-2023)
    "DIS_TOTAL": "Total count of natural disasters (2000-2023)",
    "DIS_DECLAR": "Total count of declared natural disasters (2000-2023)",
    "DIS_EQK": "Big Earthquake count (2000-2023)",
    "DIS_FIRE": "Big Fire count (2000-2023)",
    "DIS_FLD": "Big Flood count (2000-2023)",
    "DIS_STM": "Big Storm count (2000-2023)",
    "DIS_VOL": "Big Volcano count (2000-2023)",
    "DIS_TOTAL_DEATHS": "Total deaths (2000-2023)",
    "DIS_TOTAL_DAMAGE_USD": "Total damage USD (2000-2023)",
    "DIS_AVG_DEATHS": "Average deaths per disaster (2000-2023)",
    "DIS_AVG_DAMAGE_USD": "Average damage USD per disaster (2000-2023)",
    "DIS_TOTAL_DAMAGE_USD_HISTORY": "Total damage USD (1900-1999)",
    "DIS_TOTAL_DEATHS_HISTORY": "Total deaths (1900-1999)",

    # Building vulnerability indicators - Cost weighted (default)
    "VULN_DEFAULT_FRAGILE": "Percentage of buildings fragile (Default, cost-weighted) - %",
    "VULN_DEFAULT_MEDIAN": "Percentage of buildings median (Default, cost-weighted) - %",
    "VULN_DEFAULT_ROBUST": "Percentage of buildings robust (Default, cost-weighted) - %",
    "VULN_DEFAULT_ROBUSTNESS": "Default Robustness Index (Median + Robust, cost-weighted)",
    "VULN_DEFAULT_VULNERABILITY": "Default Vulnerability Index (Fragile, cost-weighted)",
    "VULN_DEFAULT_ROBUST_ONLY": "Default Pure Robustness (Robust only, cost-weighted)",
    
    "VULN_EARTHQUAKE_FRAGILE": "Percentage of buildings fragile to Earthquake (cost-weighted) - %",
    "VULN_EARTHQUAKE_MEDIAN": "Percentage of buildings median to Earthquake (cost-weighted) - %",
    "VULN_EARTHQUAKE_ROBUST": "Percentage of buildings robust to Earthquake (cost-weighted) - %",
    "VULN_EARTHQUAKE_ROBUSTNESS": "Earthquake Robustness Index (Median + Robust, cost-weighted)",
    "VULN_EARTHQUAKE_VULNERABILITY": "Earthquake Vulnerability Index (Fragile, cost-weighted)",
    "VULN_EARTHQUAKE_ROBUST_ONLY": "Earthquake Pure Robustness (Robust only, cost-weighted)",
    
    "VULN_WIND_FRAGILE": "Percentage of buildings fragile to Wind (cost-weighted) - %",
    "VULN_WIND_MEDIAN": "Percentage of buildings median to Wind (cost-weighted) - %",
    "VULN_WIND_ROBUST": "Percentage of buildings robust to Wind (cost-weighted) - %",
    "VULN_WIND_ROBUSTNESS": "Wind Robustness Index (Median + Robust, cost-weighted)",
    "VULN_WIND_VULNERABILITY": "Wind Vulnerability Index (Fragile, cost-weighted)",
    "VULN_WIND_ROBUST_ONLY": "Wind Pure Robustness (Robust only, cost-weighted)",
    
    "VULN_WATER_ROBUSTNESS": "Integrated Water Hazards Robustness Index (cost-weighted)",
    "VULN_WATER_VULNERABILITY": "Integrated Water Hazards Vulnerability Index (cost-weighted)",
    
    # Building vulnerability indicators - Dwelling weighted (_WD suffix)
    "VULN_DEFAULT_FRAGILE_WD": "Percentage of buildings fragile (Default, dwelling-weighted) - %",
    "VULN_DEFAULT_MEDIAN_WD": "Percentage of buildings median (Default, dwelling-weighted) - %",
    "VULN_DEFAULT_ROBUST_WD": "Percentage of buildings robust (Default, dwelling-weighted) - %",
    "VULN_DEFAULT_ROBUSTNESS_WD": "Default Robustness Index (Median + Robust, dwelling-weighted)",
    "VULN_DEFAULT_VULNERABILITY_WD": "Default Vulnerability Index (Fragile, dwelling-weighted)",
    "VULN_DEFAULT_ROBUST_ONLY_WD": "Default Pure Robustness (Robust only, dwelling-weighted)",
    
    "VULN_EARTHQUAKE_FRAGILE_WD": "Percentage of buildings fragile to Earthquake (dwelling-weighted) - %",
    "VULN_EARTHQUAKE_MEDIAN_WD": "Percentage of buildings median to Earthquake (dwelling-weighted) - %",
    "VULN_EARTHQUAKE_ROBUST_WD": "Percentage of buildings robust to Earthquake (dwelling-weighted) - %",
    "VULN_EARTHQUAKE_ROBUSTNESS_WD": "Earthquake Robustness Index (Median + Robust, dwelling-weighted)",
    "VULN_EARTHQUAKE_VULNERABILITY_WD": "Earthquake Vulnerability Index (Fragile, dwelling-weighted)",
    "VULN_EARTHQUAKE_ROBUST_ONLY_WD": "Earthquake Pure Robustness (Robust only, dwelling-weighted)",
    
    "VULN_WIND_FRAGILE_WD": "Percentage of buildings fragile to Wind (dwelling-weighted) - %",
    "VULN_WIND_MEDIAN_WD": "Percentage of buildings median to Wind (dwelling-weighted) - %",
    "VULN_WIND_ROBUST_WD": "Percentage of buildings robust to Wind (dwelling-weighted) - %",
    "VULN_WIND_ROBUSTNESS_WD": "Wind Robustness Index (Median + Robust, dwelling-weighted)",
    "VULN_WIND_VULNERABILITY_WD": "Wind Vulnerability Index (Fragile, dwelling-weighted)",
    "VULN_WIND_ROBUST_ONLY_WD": "Wind Pure Robustness (Robust only, dwelling-weighted)",
    
    "VULN_WATER_ROBUSTNESS_WD": "Integrated Water Hazards Robustness Index (dwelling-weighted)",
    "VULN_WATER_VULNERABILITY_WD": "Integrated Water Hazards Vulnerability Index (dwelling-weighted)",
}

# Save metadata as JSON
with open(metadata_file_path_final, 'w', encoding='utf-8') as f:
    json.dump(COLUMN_METADATA_FINAL, f, ensure_ascii=False, indent=4)

print(f"✅ Metadata JSON file has been exported.")
print(f"   Path: {metadata_file_path_final}")


# ============================================================================
# FINAL SUMMARY
# ============================================================================

print("\n" + "="*70)
print("                   ✅ PIPELINE COMPLETE!")
print("="*70)
print(f"\n📊 Final Dataset Statistics:")
print(f"   • Total countries: {len(df_final_for_analysis)}")
print(f"   • Total indicators: {len(df_final_for_analysis.columns)}")
print(f"   • World Bank indicators: {len([c for c in df_final_for_analysis.columns if c.startswith('WB_')])}")
print(f"   • MPI indicators: {len([c for c in df_final_for_analysis.columns if c.startswith('MPI_')])}")
print(f"   • Disaster indicators: {len([c for c in df_final_for_analysis.columns if c.startswith('DIS_')])}")
print(f"   • Vulnerability indicators: {len([c for c in df_final_for_analysis.columns if c.startswith('VULN_')])}")
print(f"\n📁 Output Files:")
print(f"   • CSV: {export_final_analysis_path}")
print(f"   • Metadata: {metadata_file_path_final}")
print("="*70 + "\n")



GLOBAL ANALYSIS DATA INTEGRATION PIPELINE

Reference country list (World Bank): 217 countries

--- 📊 Indicator Counts After Cleaning ---
1. WB (Reference): 44 columns
2. Disaster: 24 columns
3. MPI: 23 columns
4. Vulnerability: 31 columns

--- 🔗 Performing Fuzzy Matching ---

--- 📝 Creating Traceability Information ---

--- 🧹 Final Data Cleaning ---

             🌍 DATA INTEGRATION RESULTS SUMMARY 🌍
**World Bank Reference Country Count:** 217 countries

--- 1. Disaster Data ---
Original data country count (after standardization): 222
Successfully matched countries: 198
Matching rate: 89.19%
Unmatched countries: 24

🔥 Unmatched country list (Disaster):
['ANGUILLA', 'AZORES ISLANDS', 'CANARY ISLANDS', 'COOK ISLANDS', 'CZECHOSLOVAKIA', 'CÔTE D’IVOIRE', 'FRENCH GUIANA', 'GERMAN DEMOCRATIC REPUBLIC', 'GUADELOUPE', 'MARTINIQUE', 'MONTSERRAT', 'NETHERLANDS ANTILLES', 'NIUE', "PEOPLE'S DEMOCRATIC REPUBLIC OF YEMEN", 'RÉUNION', 'SAINT BARTHÉLEMY', 'SAINT HELENA', 'SERBIA MONTENEGRO', 'SOVIET U

# OLD

In [1]:
import pandas as pd
import os
import numpy as np
from rapidfuzz import fuzz, process
from rapidfuzz.utils import default_process
import re
import json
# ----------------------------------------------------
# file path 
# ----------------------------------------------------
folder_path = r"C:\Users\wb512463\OneDrive - WBG\Resilient Housing\Global analysis\data_processed"
raw_data_folder = r"C:\Users\wb512463\OneDrive - WBG\Resilient Housing\Global analysis\data_raw\World Bank"

# 基準となるWorld Bankデータファイル
file_wb_base = os.path.join(folder_path, "Country_WorldBank_Data_Cleaned_For_Merge.csv")

# 各ファイルのフルパス
file_disaster = os.path.join(folder_path, "country_disaster_summary_FINAL_PERIOD_SPLIT.csv")
file_mpi = os.path.join(folder_path, "country_MPI.csv")
file_vulnerability = os.path.join(folder_path, "country_vulnerability_class_shares.csv")

# ----------------------------------------------------
# Fuzzy matching roles (adjusted after several trial)
# ----------------------------------------------------
SCORE_THRESHOLD = 90

# Not allowing the following automatic matching 
PROHIBITED_PAIRS = [
    ('Arunachal Pradesh', 'Aruba'), ('Aruba', 'Arunachal Pradesh'), 
    ('Burundi', 'Brunei Darussalam'), ('Brunei', 'Burundi'), 
    ('Guatemala', 'Malta'), ('Malta', 'Guatemala'),
    ('Guinea', 'Equatorial Guinea'), ('Equatorial Guinea', 'Guinea'), 
    ('Qatar', 'Gibraltar'), ('Gibraltar', 'Qatar'),
    ('Taiwan', 'Tanzania'), ('Taiwan', 'Togo'),
]

# Standardize country names 
STANDARDIZATION_MAP = {
    "Viet Nam": "Viet Nam", "Vietnam": "Viet Nam", 
    "Côte d'Ivoire": "Cote d'Ivoire", "Ivory Coast": "Cote d'Ivoire",
    "Lao PDR": "Lao PDR", "Laos": "Lao PDR", "Lao People's Democratic Republic": "Lao PDR",
    "The former Yugoslav Republic of Macedonia": "North Macedonia", "North_Macedonia": "North Macedonia", 
    "Turkey": "Turkiye", 
    "Russia": "Russian Federation",
    "South Korea": "Korea, Rep.", "Republic of Korea": "Korea, Rep.", 
    "North Korea": "Korea, Dem. People's Rep.", "Dem People's Rep of Korea": "Korea, Dem. People's Rep.",
    "United Republic of Tanzania": "Tanzania",
    "Swaziland": "Eswatini",
    "Slovakia": "Slovak Republic",
    "Syria": "Syrian Arab Republic",
    "Congo": "Congo, Rep.",
    "Democratic Republic of the Congo": "Congo, Dem. Rep.", "Congo, Democratic Republic of the": "Congo, Dem. Rep.",
    "U.K. of Great Britain and Northern Ireland": "United Kingdom",
    "United States of America": "United States",
    "Venezuela": "Venezuela, RB",
    "US Virgin Islands": "Virgin Islands (U.S.)",
    "Saint Kitts and Nevis": "St. Kitts and Nevis",
    "Saint Lucia": "St. Lucia",
    "Saint Vincent and the Grenadines": "St. Vincent and the Grenadines",
    "Palestine, State of": "West Bank and Gaza", "Palestine": "West Bank and Gaza",
    "Bahamas": "Bahamas, The",
    "Cape Verde": "Cabo Verde",
    "Czech Republic": "Czechia",
    "Egypt": "Egypt, Arab Rep.", # 'Egypt' -> 'Egypt, Arab Rep.'
    "Gambia": "Gambia, The", # 'Gambia' -> 'Gambia, The'
    "Hong Kong": "Hong Kong SAR, China",
    "Iran  (Islamic Republic of)": "Iran, Islamic Rep.",
    "Iran": "Iran, Islamic Rep.",
    "IRAN (ISLAMIC REPUBLIC OF)":"Iran, Islamic Rep.",
    "Kyrgyzstan": "Kyrgyz Republic",
    "Macau": "Macao SAR, China",
    "Macao": "Macao SAR, China",
    "Micronesia (Federated States of)": "Micronesia, Fed. Sts.",
    "Micronesia": "Micronesia, Fed. Sts.",
    "Moldova, Republic of": "Moldova",
    "Puerto Rico": "Puerto Rico (US)",
    "United States Virgin Islands": "Virgin Islands (U.S.)",
    "Brunei": "Brunei Darussalam", 
    "Yemen": "Yemen, Rep.",
    "Somalia": "Somalia, Fed. Rep.",
    "Chad": "Chad", 
    "Gibraltar": "Gibraltar",
    "Cook Islands": "Cook Islands", 
    "Montserrat": "Montserrat",
    "Anguilla": "Anguilla",
    "Niue": "Niue",    
    "Taiwan": "Taiwan",
    "Bolivia (Plurinational State of)": "Bolivia", 
    "China, Hong Kong Special Administrative Region": "Hong Kong SAR, China",
    "Iran (Islamic Republic of)": "Iran, Islamic Rep.",
    "Democratic People's Republic of Korea": "Korea, Dem. People's Rep.",
    "China, Macao Special Administrative Region": "Macao SAR, China",
    "Saint Martin (French Part)": "St. Martin (French part)",
    "Türkiye": "Turkiye",
    "United Kingdom of Great Britain and Northern Ireland": "United Kingdom",
    "Venezuela (Bolivarian Republic of)": "Venezuela, RB",
    "State of Palestine": "West Bank and Gaza",
    "Netherlands (Kingdom of the)": "Netherlands",
    "Serbia Montenegro": "Serbia Montenegro", 
    "Taiwan (Province of China)": "Taiwan", 
    "Yemen Arab Republic": "Yemen Arab Republic", 
    "People's Democratic Republic of Yemen": "People's Democratic Republic of Yemen",
    "Germany Federal Republic": "Germany", 
    "Canary Islands": "Canary Islands",
    "French Guiana": "French Guiana",
    "Guadeloupe": "Guadeloupe",
    "Martinique": "Martinique",
    "Réunion": "Réunion",
    "Wallis and Futuna Islands": "Wallis and Futuna Islands",
    "CÔTE D’IVOIRE": "Cote d'Ivoire",
    "REPUBLIC OF MOLDOVA": "Moldova",
}

In [2]:
# ----------------------------------------------------
# Data cleaning 
# ----------------------------------------------------

def clean_country_names(df):
    """
    Clean country names and standardize
    """
    df_reset = df.reset_index()
    index_name = df_reset.columns[0]
    
    df_reset[index_name] = df_reset[index_name].astype(str)    
    df_reset[index_name] = df_reset[index_name].str.replace(r'\s+', ' ', regex=True)
    df_reset[index_name] = df_reset[index_name].str.strip()
    
    standardization_map_lower = {k.lower(): v.lower() for k, v in STANDARDIZATION_MAP.items()}
    
    temp_series_lower = df_reset[index_name].str.lower().replace(standardization_map_lower)    
    df_reset[index_name] = temp_series_lower.str.upper()

    # Integration
    grouped = df_reset.groupby(index_name)
    df_numeric = grouped.sum(numeric_only=True)
    df_non_numeric = grouped.first(numeric_only=False).select_dtypes(include=['object', 'bool'])
    df_cleaned = df_numeric.merge(df_non_numeric, left_index=True, right_index=True, how='outer')
    
    return df_cleaned

def aggregate_disaster_data(df):
    """Gaza and west bank"""
    countries_to_sum = ['Gaza Strip', 'West Bank']
    
    if not all(c in df.index for c in countries_to_sum): return df
    
    new_palestine_data = df.loc[countries_to_sum].sum(axis=0, numeric_only=True)
    df.loc['West Bank and Gaza'] = new_palestine_data
    df = df.drop(index=countries_to_sum)
    
    return df

# ----------------------------------------------------
#  Fuzzy matching 
# ----------------------------------------------------
def fuzzy_match_country_v2(country_name, choices, threshold=SCORE_THRESHOLD):
    """Fuzzy matching using country name"""
    if pd.isna(country_name): return (None, 0)
    if country_name in choices: return (country_name, 100)
    
    processed_choices = {c: default_process(c) for c in choices}
    
    match = process.extractOne(
        country_name, 
        choices, 
        scorer=fuzz.ratio, 
        score_cutoff=threshold, 
        processor=default_process 
    )
    
    if match:
        matched_country = match[0]
        score = match[1]
        
        if (country_name, matched_country) in PROHIBITED_PAIRS:
            return (None, 0)
            
        if score >= 95:
            return (matched_country, score)
        return (None, score) 
    else:
        return (None, 0)

In [3]:
def process_vulnerability_data(file_path: str):
    """Vulnerabilityファイルの特殊なマルチヘッダー構造を処理し、クリーンなDFを返します。"""
    
    # 1. 最初の4行を読み込み、ヘッダー情報を取得
    df_raw_headers = pd.read_csv(file_path, header=None, nrows=3)
    
    # 2. 実際のデータフレームを読み込み (ヘッダーは3行目=インデックス2を使用)
    df_data = pd.read_csv(file_path, header=2) # header=2 は3行目を列名として読み込む
    
    # 3. 新しい列名の生成
    new_columns = []
    
    # A列とB列 (iso3, country)
    new_columns.append('iso3')
    new_columns.append('CountryName') # country を CountryName に統一
    
    # C列以降の処理: 1行目と2行目のヘッダー情報を結合
    # df_raw_headersのインデックスは 0, 1, 2。C列以降はインデックス2から始まる
    for col_index in range(2, len(df_data.columns)):
        col_name_3rd_row = df_data.columns[col_index] # 3行目の名前 (例: default, resilient)
        
        # 1行目と2行目のヘッダーを取得 (NaNは空文字列として扱う)
        header_row_0 = str(df_raw_headers.iloc[0, col_index]).strip()
        header_row_1 = str(df_raw_headers.iloc[1, col_index]).strip()
        
        # 結合ロジック: NaNや空文字列を無視し、有効なパーツを '_' で結合
        parts = [p for p in [header_row_0, header_row_1, col_name_3rd_row] if p != 'nan' and p]
        
        # 最終的な列名
        final_col_name = '_'.join(parts).replace(' ', '_').replace('.', '_')
        new_columns.append(final_col_name)

    # 列名リストがデータ列数と一致するか確認
    if len(new_columns) != len(df_data.columns):
        raise ValueError("生成された列名数とデータ列数が一致しません。ファイル構造を確認してください。")

    # 4. 新しい列名を適用
    df_data.columns = new_columns
    
    # 5. クレンジングとインデックス設定
    df_data = df_data.set_index('CountryName')
    df_data = clean_country_names(df_data)
    
    return df_data

In [4]:
# ----------------------------------------------------
# 📌 データの読み込み、クレンジング、基準国の設定
# ----------------------------------------------------

# 0. World Bank基準データの読み込み
df_wb_base = pd.read_csv(file_wb_base).rename(columns={'CountryName': 'CountryName'}).set_index('CountryName')
base_countries = df_wb_base.index.tolist() # World Bankの国名リストを基準として設定
print(f"基準国リスト (World Bank): {len(base_countries)} カ国")


# 1. df_disaster
df_disaster = pd.read_csv(file_disaster).rename(columns={'country_name': 'CountryName', 'CountryName': 'CountryName'}).set_index('CountryName')
df_disaster = aggregate_disaster_data(df_disaster) # パレスチナ統合
df_disaster = clean_country_names(df_disaster) # 国名標準化

# 2. df_mpi 
df_mpi = pd.read_csv(file_mpi).rename(columns={'Country': 'CountryName'}).set_index('CountryName')
df_mpi = clean_country_names(df_mpi) # 国名標準化

# 3. df_vulnerability (マルチヘッダー処理を含む)
df_vulnerability = process_vulnerability_data(file_vulnerability)


# デバッグ用チェック
print("\n--- 📊 クレンジング後のインディケーター数 ---")
print(f"1. WB (基準): {len(df_wb_base.columns)} 列")
print(f"2. Disaster: {len(df_disaster.columns)} 列")
print(f"3. MPI: {len(df_mpi.columns)} 列")
print(f"4. Vulnerability: {len(df_vulnerability.columns)} 列")

基準国リスト (World Bank): 217 カ国

--- 📊 クレンジング後のインディケーター数 ---
1. WB (基準): 44 列
2. Disaster: 24 列
3. MPI: 23 列
4. Vulnerability: 16 列


In [5]:
# ----------------------------------------------------
# 📌 マッチングの実行と統合データセットの作成
# ----------------------------------------------------

# Disaster マッチング
df_disaster_reset = df_disaster.reset_index()
df_disaster_reset[['Matched_CountryName', 'Match_Score_DIS']] = df_disaster_reset['CountryName'].apply(
    lambda x: pd.Series(fuzzy_match_country_v2(x, base_countries))
)
df_disaster_matched = df_disaster_reset[df_disaster_reset['Matched_CountryName'].notna()].set_index('Matched_CountryName').drop(columns=['CountryName'])

# MPI マッチング
df_mpi_reset = df_mpi.reset_index()
df_mpi_reset[['Matched_CountryName', 'Match_Score_MPI']] = df_mpi_reset['CountryName'].apply(
    lambda x: pd.Series(fuzzy_match_country_v2(x, base_countries))
)
df_mpi_matched = df_mpi_reset[df_mpi_reset['Matched_CountryName'].notna()].set_index('Matched_CountryName').drop(columns=['CountryName'])


# Vulnerability マッチング
df_vuln_reset = df_vulnerability.reset_index()
df_vuln_reset[['Matched_CountryName', 'Match_Score_VULN']] = df_vuln_reset['CountryName'].apply(
    lambda x: pd.Series(fuzzy_match_country_v2(x, base_countries))
)
df_vuln_matched = df_vuln_reset[df_vuln_reset['Matched_CountryName'].notna()].set_index('Matched_CountryName').drop(columns=['CountryName'])


# データの最終統合 (World Bankを軸に左結合)
df_final_merged_fuzzy = df_wb_base.copy()
df_final_merged_fuzzy = df_final_merged_fuzzy.merge(df_disaster_matched, left_index=True, right_index=True, how='left')
df_final_merged_fuzzy = df_final_merged_fuzzy.merge(df_mpi_matched, left_index=True, right_index=True, how='left')
df_final_merged_fuzzy = df_final_merged_fuzzy.merge(df_vuln_matched, left_index=True, right_index=True, how='left')

# ----------------------------------------------------
# 📌 トレース情報の作成と最終データセットのエクスポート
# ----------------------------------------------------

# トレース情報の作成
df_trace = df_final_merged_fuzzy.index.to_frame(name='Merged_CountryName')

df_disaster_trace = df_disaster_reset[['CountryName', 'Matched_CountryName', 'Match_Score_DIS']].rename(columns={'CountryName': 'DISASTER_Original_Name'})
df_mpi_trace = df_mpi_reset[['CountryName', 'Matched_CountryName', 'Match_Score_MPI']].rename(columns={'CountryName': 'MPI_Original_Name'})
df_vuln_trace = df_vuln_reset[['CountryName', 'Matched_CountryName', 'Match_Score_VULN']].rename(columns={'CountryName': 'VULN_Original_Name'})

# トレース情報を結合
df_trace = df_trace.merge(df_disaster_trace, left_on='Merged_CountryName', right_on='Matched_CountryName', how='left').drop(columns=['Matched_CountryName']).drop_duplicates(subset=['Merged_CountryName'], keep='first')
df_trace = df_trace.merge(df_mpi_trace, left_on='Merged_CountryName', right_on='Matched_CountryName', how='left').drop(columns=['Matched_CountryName']).drop_duplicates(subset=['Merged_CountryName'], keep='first')
df_trace = df_trace.merge(df_vuln_trace, left_on='Merged_CountryName', right_on='Matched_CountryName', how='left').drop(columns=['Matched_CountryName']).drop_duplicates(subset=['Merged_CountryName'], keep='first')

df_trace['WB_Original_Name'] = df_trace['Merged_CountryName']
df_trace['WB_Match_Score'] = 100.0

# 最終エクスポートデータフレームの作成
df_trace = df_trace.set_index('Merged_CountryName')
df_export = df_trace.merge(df_final_merged_fuzzy, left_index=True, right_index=True, how='right')
df_export = df_export.reset_index().rename(columns={'index': 'Merged_CountryName'})

# ----------------------------------------------------
# 📌 6. 最終整理とデータ型変換
# ----------------------------------------------------
# WBデータには'Country Code'が含まれているため、残す。

columns_to_drop = [
    # 冗長なスコア列やISOコードを削除
    'ISO country numeric code', 'ISO country code',
    'ISO_CODE', 'iso3', 
]
cols_to_drop_present = [col for col in columns_to_drop if col in df_export.columns]
df_analysis = df_export.drop(columns=cols_to_drop_present, errors='ignore')



In [6]:
# ----------------------------------------------------
# 📌 統合結果の集計 (変数の定義と計算)
# ----------------------------------------------------

# A. World Bank (基準) の国数
# base_countriesはdf_wb_baseのインデックスから生成されている必要があります
wb_count = len(base_countries)

# B. Disasterの元の国数 (標準化後)
# df_disasterはclean_country_names適用後のデータフレームです
disaster_original_count = len(df_disaster.index)
# C. Disasterの統合された国数 (マッチング後)
# df_disaster_matchedはマッチングが成功した行のみを含むデータフレームです
disaster_matched_count = len(df_disaster_matched)

# D. MPIの元の国数 (標準化後)
mpi_original_count = len(df_mpi.index)
# E. MPIの統合された国数 (マッチング後)
mpi_matched_count = len(df_mpi_matched)

# F. Vulnerabilityの元の国数 (標準化後)
vuln_original_count = len(df_vulnerability.index)
# G. Vulnerabilityの統合された国数 (マッチング後)
vuln_matched_count = len(df_vuln_matched)

# マッチしなかった国名のリスト
# df_*_resetはマッチング前の国名とマッチング結果を含むデータフレームです
disaster_unmatched_list = df_disaster_reset[df_disaster_reset['Matched_CountryName'].isna()]['CountryName'].tolist()
mpi_unmatched_list = df_mpi_reset[df_mpi_reset['Matched_CountryName'].isna()]['CountryName'].tolist()
vuln_unmatched_list = df_vuln_reset[df_vuln_reset['Matched_CountryName'].isna()]['CountryName'].tolist()


# ----------------------------------------------------
# 📌 結果の出力
# ----------------------------------------------------

print("\n" + "="*50)
print("             🌍 データ統合結果サマリー 🌍")
print("="*50)
print(f"**World Bank 基準国数:** {wb_count} カ国\n")

# Disaster
print("--- 1. Disaster データ ---")
print(f"元データ国数 (標準化後): {disaster_original_count}")
print(f"マッチング成功国数: {disaster_matched_count}")
print(f"マッチング率: {disaster_matched_count / disaster_original_count * 100:.2f}%")
print(f"マッチしなかった国数: {disaster_original_count - disaster_matched_count}")
print("\n🔥 マッチしなかった国名リスト (Disaster):")
print(disaster_unmatched_list)
print("-" * 50)


# MPI
print("\n--- 2. MPI データ ---")
print(f"元データ国数 (標準化後): {mpi_original_count}")
print(f"マッチング成功国数: {mpi_matched_count}")
print(f"マッチング率: {mpi_matched_count / mpi_original_count * 100:.2f}%")
print(f"マッチしなかった国数: {mpi_original_count - mpi_matched_count}")
print("\n🔥 マッチしなかった国名リスト (MPI):")
print(mpi_unmatched_list)
print("-" * 50)

# Vulnerability
print("\n--- 3. Vulnerability データ ---")
print(f"元データ国数 (標準化後): {vuln_original_count}")
print(f"マッチング成功国数: {vuln_matched_count}")
print(f"マッチング率: {vuln_matched_count / vuln_original_count * 100:.2f}%")
print(f"マッチしなかった国数: {vuln_original_count - vuln_matched_count}")
print("\n🔥 マッチしなかった国名リスト (Vulnerability):")
print(vuln_unmatched_list)
print("-" * 50)


print("\n" + "="*50)
print(f"**最終統合データセットの行数:** {len(df_final_merged_fuzzy)} 行 (World Bank基準)")
print("="*50)


             🌍 データ統合結果サマリー 🌍
**World Bank 基準国数:** 217 カ国

--- 1. Disaster データ ---
元データ国数 (標準化後): 222
マッチング成功国数: 199
マッチング率: 89.64%
マッチしなかった国数: 23

🔥 マッチしなかった国名リスト (Disaster):
['ANGUILLA', 'AZORES ISLANDS', 'CANARY ISLANDS', 'COOK ISLANDS', 'CZECHOSLOVAKIA', 'FRENCH GUIANA', 'GERMAN DEMOCRATIC REPUBLIC', 'GUADELOUPE', 'MARTINIQUE', 'MONTSERRAT', 'NETHERLANDS ANTILLES', 'NIUE', "PEOPLE'S DEMOCRATIC REPUBLIC OF YEMEN", 'RÉUNION', 'SAINT BARTHÉLEMY', 'SAINT HELENA', 'SERBIA MONTENEGRO', 'SOVIET UNION', 'TAIWAN', 'TOKELAU', 'WALLIS AND FUTUNA ISLANDS', 'YEMEN ARAB REPUBLIC', 'YUGOSLAVIA']
--------------------------------------------------

--- 2. MPI データ ---
元データ国数 (標準化後): 109
マッチング成功国数: 109
マッチング率: 100.00%
マッチしなかった国数: 0

🔥 マッチしなかった国名リスト (MPI):
[]
--------------------------------------------------

--- 3. Vulnerability データ ---
元データ国数 (標準化後): 215
マッチング成功国数: 207
マッチング率: 96.28%
マッチしなかった国数: 8

🔥 マッチしなかった国名リスト (Vulnerability):
['ANGUILLA', 'COOK ISLANDS', 'FRENCH GUIANA', 'GUADELOUPE', 'MARTINI

In [7]:
# ----------------------------------------------------
# 📌 1. 列名クリーニング関数の定義
# ----------------------------------------------------

def clean_dataframe_columns(df):
    """
    データ分析を妨げる不必要なサフィックスや情報を列名から削除し、統一的なプレフィックスを適用します。
    """
    new_columns_map = {}
    
    # 冗長なサフィックスを削除するための正規表現
    UNWANTED_SUFFIX_PATTERN = r'_UNNAMED:\_\d+$|_UNNAMED'
    
    for col in df.columns:
        new_col = col.upper() # 全て大文字化
        
        # 1. 不必要なサフィックスの削除 (例: _UNNAMED:_9, _UNNAMED)
        new_col = re.sub(UNWANTED_SUFFIX_PATTERN, '', new_col)
        
        # 2. 特殊な文字やスペースの置換
        new_col = new_col.replace(' ', '_').replace('-', '_').replace(':', '').replace('.', '_').replace('__', '_')
        
        # 3. プレフィックスの適用と明確な名称への変換
        
        if new_col == 'MERGED_COUNTRYNAME' or new_col == 'COUNTRYNAME':
            new_col = 'COUNTRY_NAME'
        elif new_col == 'COUNTRY_CODE':
            new_col = 'WB_COUNTRY_CODE'

        # WB経済指標
        elif new_col.startswith(('GDP_', 'POVERTY_', 'GINI_', 'SLUM_POPULATION_')):
            if not new_col.startswith('WB_'):
                 new_col = 'WB_' + new_col
        elif new_col == 'GNI_PC':
            new_col = 'WB_GNI_PC'
        
        # MPI指標の明確化 (重複を避けるため全て再定義)
        elif new_col in ['HEADCOUT_RATIO', 'INTENSITY_OF_DEPREVIATION', 'POPULATION_2023_K', 'YEARS_OF_SCHOOLING', 
                         'SANITATION', 'DRINKING_WATER', 'ELECTRICITY', 'ASSETS', 'NUTRITION', 
                         'CHILD_MORTALITY', 'SCHOOL_ATTENDANCE', 'COOKING_FUEL', 'HOUSING', 
                         'MONETARY_POVERTY_USD3', 'HID', 'WORLD_REGION', 'SURVEY', 'YEAR', 'INCOME_CATEGORY']:
            new_col = 'MPI_' + new_col
            # 'MONETARY_POVERTY__USD3' の修正
            if new_col == 'MPI_MONETARY_POVERTY_USD3':
                new_col = 'MPI_POV_USD3'
            
        # Vulnerability指標の明確化 (VULN)
        elif new_col.startswith(('DEFAULT_', 'EARTHQUAKE_', 'WIND_', 'FLOOD_', 'STORM_SURGE_', 'TSUNAMI_')):
            if not new_col.startswith('VULN_'):
                new_col = 'VULN_' + new_col
            
        # Disaster指標 (DIS)
        elif new_col.startswith('DIS_'):
            pass

        # 最終的な重複チェックとクリーンアップ (万が一のために)
        if new_col in new_columns_map.values():
            new_col += '_DUPLICATE'

        new_columns_map[col] = new_col

    # 列名の適用
    df = df.rename(columns=new_columns_map)
    return df

# ----------------------------------------------------
# 📌 2. データの処理とCSVエクスポート
# ----------------------------------------------------

# A. マッチング関連列の削除
COLUMNS_TO_DROP_FOR_ANALYSIS = [
    "WB_Original_Name", "WB_Match_Score", "ISO_CODE",
    "DISASTER_Original_Name", "Match_Score_DIS", "Match_Score_DIS_x", "Match_Score_DIS_y",
    "MPI_Original_Name", "Match_Score_MPI", "Match_Score_MPI_x", "Match_Score_MPI_y",
    "VULN_Original_Name", "Match_Score_VULN", "Match_Score_VULN_x", "Match_Score_VULN_y",
]

# 実際にデータフレームに存在する列のみを削除対象とする
cols_to_drop_present = [col for col in COLUMNS_TO_DROP_FOR_ANALYSIS if col in df_analysis.columns]

# マッチング関連列の削除
df_final_for_analysis = df_analysis.drop(columns=cols_to_drop_present, errors='ignore')

# B. 列名クリーニングの適用
df_final_for_analysis = clean_dataframe_columns(df_final_for_analysis)

In [8]:

# Creating Variables for VULN 

# 0. 定数定義
HAZARDS_TO_INTEGRATE = ['FLOOD', 'STORM_SURGE', 'TSUNAMI']
INTEGRATED_WATER_HAZARD = 'WATER'

# VULN列のみを抽出し、コピーを作成
vuln_cols_raw = [col for col in df_final_for_analysis.columns if col.startswith('VULN_')]
df_vuln_processed = df_final_for_analysis[vuln_cols_raw].copy()

# 1. VULN列を確実に数値型に変換
for col in vuln_cols_raw:
    df_vuln_processed[col] = pd.to_numeric(df_vuln_processed[col], errors='coerce')


# 2. 集計インデックスの計算 (存在しない _ROBUST 列にも対応)
processed_types = set()
# 元のデータに存在するハザードタイプを抽出
vuln_pattern = re.compile(r'VULN_([A-Z_]+)_(FRAGILE|MEDIAN|ROBUST)$')
hazard_types_available = set()
for col in df_vuln_processed.columns:
    match = vuln_pattern.match(col)
    if match:
        hazard_types_available.add(match.group(1))

# 存在する全てのハザードタイプについて集計列を作成
for risk_type in sorted(list(hazard_types_available)):
    if risk_type in processed_types:
        continue

    fragile_col = f'VULN_{risk_type}_FRAGILE'
    median_col = f'VULN_{risk_type}_MEDIAN'
    robust_col = f'VULN_{risk_type}_ROBUST'

    # 2a. ROBUST_ONLY (純粋な堅牢性) 変数: ROBUST のみ
    robust_only_col_name = f'VULN_{risk_type}_ROBUST_ONLY'
    if robust_col in df_vuln_processed.columns:
        df_vuln_processed[robust_only_col_name] = df_vuln_processed[robust_col]

    # 2b. ROBUSTNESS (頑健性指数) 変数: MEDIAN + ROBUST
    robustness_col_name = f'VULN_{risk_type}_ROBUSTNESS'
    
    # 頑健性指数を計算するための列リスト（存在する列のみ）
    robustness_parts = []
    if median_col in df_vuln_processed.columns:
        robustness_parts.append(df_vuln_processed[median_col])
    if robust_col in df_vuln_processed.columns:
        robustness_parts.append(df_vuln_processed[robust_col])

    if robustness_parts:
        # 存在する列の合計を計算 (NaNは伝播するため、元の欠損値は維持される)
        # FLOOD/STORM_SURGE/TSUNAMIではROBUST列がないため、MEDIAN列のみの合計となる
        df_vuln_processed[robustness_col_name] = sum(robustness_parts)
    # 少なくとも MEDIAN または ROBUST が存在しない場合は、この列は作成されない

    # 2c. VULNERABILITY (脆弱性指数) 変数: FRAGILE only
    vulnerability_col_name = f'VULN_{risk_type}_VULNERABILITY'
    if fragile_col in df_vuln_processed.columns:
        df_vuln_processed[vulnerability_col_name] = df_vuln_processed[fragile_col]

    processed_types.add(risk_type)
print(f"Calculated aggregated indices for: {processed_types}")


# 3. Water Hazardsの統合 (FLOOD, STORM_SURGE, TSUNAMIの平均)
integrated_robustness_col = f'VULN_{INTEGRATED_WATER_HAZARD}_ROBUSTNESS'
integrated_vulnerability_col = f'VULN_{INTEGRATED_WATER_HAZARD}_VULNERABILITY'

# 3a. ROBUSTNESSの統合
robustness_cols_to_avg = [f'VULN_{h}_ROBUSTNESS' for h in HAZARDS_TO_INTEGRATE]
# 💡 FIX: df_vuln_processedに存在する列のみを選択してKeyErrorを回避
existing_robustness_cols = [col for col in robustness_cols_to_avg if col in df_vuln_processed.columns]

if existing_robustness_cols:
    # 存在する集計列の平均を計算
    df_vuln_processed[integrated_robustness_col] = df_vuln_processed[existing_robustness_cols].mean(axis=1)
else:
    # 統合の元となる列が一つもない場合は、全てNaNの列を作成
    df_vuln_processed[integrated_robustness_col] = np.nan

# 3b. VULNERABILITYの統合
vulnerability_cols_to_avg = [f'VULN_{h}_VULNERABILITY' for h in HAZARDS_TO_INTEGRATE]
# 💡 FIX: df_vuln_processedに存在する列のみを選択してKeyErrorを回避
existing_vulnerability_cols = [col for col in vulnerability_cols_to_avg if col in df_vuln_processed.columns]

if existing_vulnerability_cols:
    # 存在する集計列の平均を計算
    df_vuln_processed[integrated_vulnerability_col] = df_vuln_processed[existing_vulnerability_cols].mean(axis=1)
else:
    # 統合の元となる列が一つもない場合は、全てNaNの列を作成
    df_vuln_processed[integrated_vulnerability_col] = np.nan


# 4. 統合されたハザードの元の集計列を削除 (重複排除のため)
cols_to_drop_from_main = []
all_hazard_types = hazard_types_available.union(set(HAZARDS_TO_INTEGRATE))

for hazard in all_hazard_types:
    # 元の構成要素 (FRAGILE, MEDIAN, ROBUST) を削除
    for suffix in ['_FRAGILE', '_MEDIAN', '_ROBUST']:
        cols_to_drop_from_main.append(f'VULN_{hazard}{suffix}')
    
    # 統合されたハザード (FLOODなど) の集計指標も削除
    if hazard in HAZARDS_TO_INTEGRATE:
        for suffix in ['_ROBUSTNESS', '_VULNERABILITY', '_ROBUST_ONLY']:
             cols_to_drop_from_main.append(f'VULN_{hazard}{suffix}')

# 重複を削除し、df_analysisに存在する列のみを対象
cols_to_drop_final = [col for col in list(set(cols_to_drop_from_main)) if col in df_final_for_analysis.columns]
df_final_for_analysis.drop(columns=cols_to_drop_final, errors='ignore', inplace=True)
print(f"Dropped {len(cols_to_drop_final)} old/component VULN columns from df_analysis.")


# ----------------------------------------------------
# 📌 VULN集計指標のメインデータフレームへの統合 (Persistence)
# ----------------------------------------------------
print("--- 🔧 VULN集計指標のメインデータフレームへの統合 ---")
# df_vuln_processed に含まれる新しいVULN指標を df_analysis に追加します。
new_agg_cols_pattern = r'VULN_[A-Z_]+_(ROBUSTNESS|VULNERABILITY|ROBUST_ONLY)$'
cols_to_merge = [col for col in df_vuln_processed.columns if re.match(new_agg_cols_pattern, col)]

# df_analysisに存在しない列のみを追加
new_cols_added = []
for col in cols_to_merge:
    if col not in df_final_for_analysis.columns:
        df_final_for_analysis[col] = df_vuln_processed[col]
        new_cols_added.append(col)
        

print(f"Merged {len(new_cols_added)} new aggregated VULN indices into df_analysis: {new_cols_added}")

Calculated aggregated indices for: {'WIND', 'EARTHQUAKE', 'STORM_SURGE', 'FLOOD', 'DEFAULT', 'TSUNAMI'}
Dropped 15 old/component VULN columns from df_analysis.
--- 🔧 VULN集計指標のメインデータフレームへの統合 ---
Merged 17 new aggregated VULN indices into df_analysis: ['VULN_DEFAULT_ROBUST_ONLY', 'VULN_DEFAULT_ROBUSTNESS', 'VULN_DEFAULT_VULNERABILITY', 'VULN_EARTHQUAKE_ROBUST_ONLY', 'VULN_EARTHQUAKE_ROBUSTNESS', 'VULN_EARTHQUAKE_VULNERABILITY', 'VULN_FLOOD_ROBUSTNESS', 'VULN_FLOOD_VULNERABILITY', 'VULN_STORM_SURGE_ROBUSTNESS', 'VULN_STORM_SURGE_VULNERABILITY', 'VULN_TSUNAMI_ROBUSTNESS', 'VULN_TSUNAMI_VULNERABILITY', 'VULN_WIND_ROBUST_ONLY', 'VULN_WIND_ROBUSTNESS', 'VULN_WIND_VULNERABILITY', 'VULN_WATER_ROBUSTNESS', 'VULN_WATER_VULNERABILITY']


In [9]:




# ----------------------------------------------------
# 📌 3. 新しいCSVとしてエクスポート
# ----------------------------------------------------
export_final_analysis_path = os.path.join(folder_path, "global_analysis_final_for_modeling.csv")
df_final_for_analysis.to_csv(export_final_analysis_path, index=False, encoding='utf-8-sig')

print("\n--- 💾 CSVファイルのエクスポート完了 ---")
print(f"🎉 分析・モデリング用の最終CSVファイルがエクスポートされました。")
print(f"パス: {export_final_analysis_path}")
print(f"最終的なデータ形状: {df_final_for_analysis.shape}")


--- 💾 CSVファイルのエクスポート完了 ---
🎉 分析・モデリング用の最終CSVファイルがエクスポートされました。
パス: C:\Users\wb512463\OneDrive - WBG\Resilient Housing\Global analysis\data_processed\global_analysis_final_for_modeling.csv
最終的なデータ形状: (217, 107)


## add json for indicator name

In [10]:
# ----------------------------------------------------
# 📌 4. 分析用JSONメタデータの作成とエクスポート
# ----------------------------------------------------
metadata_file_path_final = os.path.join(folder_path, "column_metadata_final_for_analysis.json")

COLUMN_METADATA_FINAL = {
    # ------------------
    # Country Key & WB Base
    # ------------------
    "COUNTRY_NAME": "Country Name (World Bank Standardized Name) - Key Column",
    "WB_COUNTRY_CODE": "Country Code (e.g., USA, JPN) - Categorical",
    "WB_GDP_PPP_Intl_2021_Value": "GDP (PPP 2021 US$)",
    "WB_GDP_PPP_Intl_2021_Year": "GDP (PPP 2021 US$) - Latest Year",
    "WB_GDP_PCAP_PPP_Intl_2021_Value": "GDP per capita (PPP 2021 US$)",
    "WB_GDP_PCAP_PPP_Intl_2021_Year": "GDP per capita (PPP 2021 US$) - Latest Year",
    "WB_GNI_PC": "Gross National Income per capita (WB Source)",
    "WB_POVERTY_SURVEY_MEAN_INCOME_PPP_VALUE": "Poverty Survey Mean Income (PPP) - Value",
    "WB_POVERTY_SURVEY_MEAN_INCOME_PPP_YEAR": "Poverty Survey Mean Income (PPP) - Year",
    "WB_POVERTY_HC_RATIO_AT_USD8_30_VALUE": "Poverty Headcount Ratio at $8.30 PPP/day - Value",
    "WB_POVERTY_HC_RATIO_AT_USD8_30_YEAR": "Poverty Headcount Ratio at $8.30 PPP/day - Year",
    "WB_POVERTY_HC_RATIO_NATIONAL_LINE_VALUE": "Poverty Headcount Ratio at National Line - Value",
    "WB_POVERTY_HC_RATIO_NATIONAL_LINE_YEAR": "Poverty Headcount Ratio at National Line - Year",
    "WB_POVERTY_HC_RATIO_AT_USD4_20_VALUE": "Poverty Headcount Ratio at $4.20 PPP/day - Value",
    "WB_POVERTY_HC_RATIO_AT_USD4_20_YEAR": "Poverty Headcount Ratio at $4.20 PPP/day - Year",
    "WB_GINI_INDEX_VALUE": "Gini Index (Inequality Measure) - Value",
    "WB_GINI_INDEX_YEAR": "Gini Index - Year",
    "WB_POVERTY_GAP_AT_USD3_00_VALUE": "Poverty Gap at $3.00 PPP/day - Value",
    "WB_POVERTY_GAP_AT_USD3_00_YEAR": "Poverty Gap at $3.00 PPP/day - Year",
    "WB_SLUM_POPULATION_URBAN_PCT_VALUE": "Slum Population as Percentage of Urban Population - Value",
    "WB_SLUM_POPULATION_URBAN_PCT_YEAR": "Slum Population as Percentage of Urban Population - Year",
    'WB_WGI_VOICE_ACCOUNTABILITY_VALUE': 'WGI_Voice_Accountability_Value',
    'WB_WGI_POLITICAL_STABILITY_VALUE': 'WGI_Political_Stability_Value',
    'WB_WGI_GOVERNMENT_EFFECTIVENESS_VALUE': 'WGI_Government_Effectiveness_Value',
    'WB_WGI_REGULATORY_QUALITY_VALUE': 'WGI_Regulatory_Quality_Value',
    'WB_WGI_RULE_OF_LAW_VALUE': 'WGI_Rule_of_Law_Value',
    'WB_WGI_CONTROL_OF_CORRUPTION_VALUE': 'WGI_Control_of_Corruption_Value',
    'WB_WGI_VOICE_ACCOUNTABILITY_YEAR': 'WGI_Voice_Accountability_Year',
    'WB_WGI_POLITICAL_STABILITY_YEAR': 'WGI_Political_Stability_Year',
    'WB_WGI_GOVERNMENT_EFFECTIVENESS_YEAR': 'WGI_Government_Effectiveness_Year',
    'WB_WGI_REGULATORY_QUALITY_YEAR': 'WGI_Regulatory_Quality_Year',
    'WB_WGI_RULE_OF_LAW_YEAR': 'WGI_Rule_of_Law_Year',
    'WB_WGI_CONTROL_OF_CORRUPTION_YEAR': 'WGI_Control_of_Corruption_Year',
    'WB_DB_CONSTRUCTION_PERMITS_SCORE' : 'Doing Business Construction Permt Score',
    'WB_DB_BUILDING_QUALITY_INDEX' : 'Doing Business Building Quality Index',
    'WB_POPULATION_TOTAL_YEAR' : 'Country Population_year',
    'WB_POPULATION_TOTAL_VALUE' : 'Country Population',

    # ------------------
    # MPI Indicators (Numeric & Categorical)
    # ------------------
    "MPI": "Multi-dimenstional Poverty Index - 2025",
    "MPI_HEADCOUT_RATIO": "Proportion of population who are multidimensionally poor - %",
    "MPI_INTENSITY_OF_DEPREVIATION": "Intensity of deprivation among the poor - %",
    "MPI_POPULATION_2023_K": "2023 Population (Unit: thousands) - Integer",
    "MPI_YEARS_OF_SCHOOLING": "Percentage of people deprived in years of schooling - %",
    "MPI_SANITATION": "Percentage of people deprived in sanitation - %",
    "MPI_DRINKING_WATER": "Percentage of people deprived in drinking water - %",
    "MPI_ELECTRICITY": "Percentage of people deprived in electricity - %",
    "MPI_ASSETS": "Percentage of people deprived in assets - %",
    "MPI_NUTRITION": "Percentage of people deprived in nutrition - %",
    "MPI_CHILD_MORTALITY": "Percentage of people deprived in child mortality - %",
    "MPI_SCHOOL_ATTENDANCE": "Percentage of people deprived in school attendance - %",
    "MPI_COOKING_FUEL": "Percentage of people deprived in cooking fuel - %",
    "MPI_HOUSING": "Percentage of people deprived in housing - %",
    "MPI_POV_USD3": "Poverty rate at $3 a day (MPI Source) - %",
    "MPI_HID": "Human Development Indicator - Value between 0.00 and 1.00",
    "MPI_WORLD_REGION": "World Region (e.g., East Asia) - Categorical Data",
    "MPI_SURVEY": "Name of the survey used for the MPI source - Categorical Data",
    "MPI_YEAR": "Calendar year of the survey used for the MPI source - Categorical Data",
    "MPI_INCOME_CATEGORY": "Country Income Category (e.g., Low Income) - Categorical Data",

    # ------------------
    # Disaster Data (DIS) - Count of natural disasters (2000-2023)
    # ------------------
    "DIS_TOTAL": "Total count of natural disasters (2000-2023)",
    "DIS_DECLAR": "Total count of declared natural disasters (2000-2023)",
    "DIS_EQK": "Big Earthquake count (2000-2023)",
    "DIS_FIRE": "Big Fire count (2000-2023)",
    "DIS_FLD": "Big Flood count (2000-2023)",
    "DIS_STM": "Big Storm count (2000-2023)",
    "DIS_VOL": "Big Volcano count (2000-2023)",
    "DIS_TOTAL_DEATHS": "Total deaths (2000-2023)",
    "DIS_TOTAL_DAMAGE_USD": "Total damage USD (2000-2023)",
    "DIS_AVG_DEATHS": "Average Death (2000-2023)",
    "DIS_AVG_DAMAGE_USD": "Aveage damage USD (2000-2023)",
    "DIS_TOTAL_DAMAGE_USD_HISTORY" :"Total damage USD (1900-1999)",
    "DIS_TOTAL_DEATHS_HISTORY": "Total deaths (1900-1999)",
    # ------------------
    # Building Vulnerability Data (VULN) - Percentage of buildings
    # ------------------
    "VULN_DEFAULT_FRAGILE": "Percentage of buildings classified as fragile (Default setting) - %",
    "VULN_DEFAULT_MEDIAN": "Percentage of buildings classified as median (Default setting) - %",
    "VULN_DEFAULT_ROBUST": "Percentage of buildings classified as robust (Default setting) - %",
    "VULN_EARTHQUAKE_FRAGILE": "Percentage of buildings fragile to Earthquake - %",
    "VULN_EARTHQUAKE_MEDIAN": "Percentage of buildings median to Earthquake - %",
    "VULN_EARTHQUAKE_ROBUST": "Percentage of buildings robust to Earthquake - %",
    "VULN_WIND_FRAGILE": "Percentage of buildings fragile to Wind - %",
    "VULN_WIND_MEDIAN": "Percentage of buildings median to Wind - %",
    "VULN_WIND_ROBUST": "Percentage of buildings robust to Wind - %",
    "VULN_FLOOD_FRAGILE": "Percentage of buildings fragile to Flood - %",
    "VULN_FLOOD_MEDIAN": "Percentage of buildings median to Flood - %",
    "VULN_STORM_SURGE_FRAGILE": "Percentage of buildings fragile to Storm surge - %",
    "VULN_STORM_SURGE_MEDIAN": "Percentage of buildings median to Storm surge - %",
    "VULN_TSUNAMI_FRAGILE": "Percentage of buildings fragile to Tsunami - %",
    "VULN_TSUNAMI_MEDIAN": "Percentage of buildings median to Tsunami - %"
}

# Save the dictionary as a JSON file
with open(metadata_file_path_final, 'w', encoding='utf-8') as f:
    json.dump(COLUMN_METADATA_FINAL, f, ensure_ascii=False, indent=4)

print("\n--- 📝 JSONファイルのエクスポート完了 ---")
print(f"🎉 分析・モデリング用の最終JSONファイルがエクスポートされました。")
print(f"パス: {metadata_file_path_final}")


--- 📝 JSONファイルのエクスポート完了 ---
🎉 分析・モデリング用の最終JSONファイルがエクスポートされました。
パス: C:\Users\wb512463\OneDrive - WBG\Resilient Housing\Global analysis\data_processed\column_metadata_final_for_analysis.json
